# Movie Financial Success and Investment Risk Prediction

CSF 3127: AI in Finance, Apurv Khanna (Reg. No. 240958208)

Dataset: TMDB 5000 Movie Dataset (Kaggle), file `tmdb_5000_movies.csv`.

## 1. Clean the data and define success

A budget or revenue of 0 means the value is missing, so those rows are removed. A film is a success when ROI (revenue / budget) is at least 2.

In [ ]:
import pandas as pd, numpy as np, json
df = pd.read_csv('tmdb_5000_movies.csv')
df = df[(df['budget'] > 0) & (df['revenue'] > 0)].copy()   # zeros mean missing data
df['release_date'] = pd.to_datetime(df['release_date'])
df = df.dropna(subset=['release_date', 'runtime'])
df['year'] = df['release_date'].dt.year
df['month'] = df['release_date'].dt.month
df['roi'] = df['revenue'] / df['budget']
df['success'] = (df['roi'] >= 2).astype(int)               # 1 = success, 0 = failure
df['genre_list'] = df['genres'].apply(lambda x: [g['name'] for g in json.loads(x)])
print(df.shape)
print(df['success'].value_counts())

## 2. Build the features

Only pre-release information is used. Votes and popularity are left out to avoid data leakage.

In [ ]:
genres = pd.get_dummies(df['genre_list'].explode()).groupby(level=0).sum()
X = pd.concat([df[['budget', 'runtime', 'year', 'month']], genres], axis=1)
y = df['success']
print(X.shape)

## 3. Train and compare two models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf = RandomForestClassifier(n_estimators=200, random_state=42)

for name, m in [('Logistic Regression', lr), ('Random Forest', rf)]:
    m.fit(X_train, y_train)
    print(name)
    print(classification_report(y_test, m.predict(X_test)))

## 4. Which features mattered most

In [ ]:
import matplotlib.pyplot as plt
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 features (Random Forest)')
plt.show()